# SSFCT — Salinas Spatial-Split Experiment

Uses the **official SSFCT implementation** from `SQlll-lang/NN-SSFCT-2025` and the project's saved spatial split.

**Configuration (matched to the previous HybridSN notebook where compatible):** seed 42, PCA-30, patch 15, batch size 128, fixed 50 epochs (no early stopping), AdamW (`lr=1e-3`, `weight_decay=1e-4`), cosine scheduler, best checkpoint by validation Macro-F1. Metrics include Accuracy, Macro-F1, per-class report, confusion matrix, parameters, model size, FLOPs when supported, inference latency, throughput, and peak GPU memory.

**Caution:** batch size 128 may exceed a 4 GB RTX 2050. If CUDA OOM occurs, lower `BATCH_SIZE` (e.g. 32 or 16), recreate the model/loaders, and note the changed batch size. This is a spatial-split adaptation of the official model, not the repository's default split protocol.

> **Corrected notebook note:** The training helper uses the metrics already imported in this notebook (`accuracy_score` and `f1_score`), adds the SSFCT channel dimension at axis 1, displays batch progress, and saves resumable checkpoints after each completed epoch.


In [1]:
# 1. Configuration and imports
from pathlib import Path
import os, sys, subprocess, time, json, random, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.io import loadmat
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, cohen_kappa_score
from tqdm.auto import tqdm

SEED = 42
PATCH_SIZE = 15
PCA_COMPONENTS = 30
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
WARMUP_RUNS = 20
TIMED_RUNS = 100

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "outputs" / "salinas" / "salinas_spatial_split_seed42.npz").exists()),
    cwd
)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "salinas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_PATH = OUTPUT_DIR / "salinas_spatial_split_seed42.npz"
print("Project root:", PROJECT_ROOT)
print("Split path:", SPLIT_PATH)


c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 2050
VRAM GB: 4.0
Project root: C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Split path: C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_spatial_split_seed42.npz


In [2]:
# 2. Load Salinas cube, labels, and the existing spatial split
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Spatial split not found: {SPLIT_PATH}. Set PROJECT_ROOT to the project root.")

cube_candidates = [
    PROJECT_ROOT / "data" / "raw" / "salinas" / "Salinas_corrected.mat",
    PROJECT_ROOT / "data" / "salinas" / "Salinas_corrected.mat",
    PROJECT_ROOT / "data" / "Salinas" / "Salinas_corrected.mat",
    PROJECT_ROOT / "Datasets" / "Salinas" / "Salinas_corrected.mat",
    PROJECT_ROOT / "datasets" / "Salinas" / "Salinas_corrected.mat",
    PROJECT_ROOT / "Salinas_corrected.mat",
]
gt_candidates = [
    PROJECT_ROOT / "data" / "raw" / "salinas" / "Salinas_gt.mat",
    PROJECT_ROOT / "data" / "salinas" / "Salinas_gt.mat",
    PROJECT_ROOT / "data" / "Salinas" / "Salinas_gt.mat",
    PROJECT_ROOT / "Datasets" / "Salinas" / "Salinas_gt.mat",
    PROJECT_ROOT / "datasets" / "Salinas" / "Salinas_gt.mat",
    PROJECT_ROOT / "Salinas_gt.mat",
]
CUBE_PATH = next((p for p in cube_candidates if p.exists()), None)
GT_PATH = next((p for p in gt_candidates if p.exists()), None)
if CUBE_PATH is None or GT_PATH is None:
    raise FileNotFoundError("Could not locate Salinas_corrected.mat and Salinas_gt.mat. Set CUBE_PATH and GT_PATH manually.")

cube_mat, gt_mat = loadmat(CUBE_PATH), loadmat(GT_PATH)
cube_key = "salinas_corrected" if "salinas_corrected" in cube_mat else next(k for k in cube_mat if not k.startswith("__") and np.asarray(cube_mat[k]).ndim == 3)
gt_key = "salinas_gt" if "salinas_gt" in gt_mat else next(k for k in gt_mat if not k.startswith("__") and np.asarray(gt_mat[k]).ndim == 2)
cube = np.asarray(cube_mat[cube_key], dtype=np.float32)
gt = np.asarray(gt_mat[gt_key], dtype=np.int64)
if cube.shape[:2] != gt.shape:
    raise ValueError(f"Cube/GT dimensions mismatch: {cube.shape} vs {gt.shape}")

split = np.load(SPLIT_PATH)
required = {"train_indices", "val_indices", "test_indices"}
if not required.issubset(split.files):
    raise KeyError(f"Split file needs {sorted(required)}; found {split.files}")
rows, cols = np.where(gt > 0)
labels_all = gt[rows, cols].astype(np.int64) - 1
train_indices = np.asarray(split["train_indices"], dtype=np.int64)
val_indices = np.asarray(split["val_indices"], dtype=np.int64)
test_indices = np.asarray(split["test_indices"], dtype=np.int64)
n = len(labels_all)
for name, idx in [("train", train_indices), ("val", val_indices), ("test", test_indices)]:
    if len(idx) and (idx.min() < 0 or idx.max() >= n):
        raise IndexError(f"{name} indices are not valid indices into the row-major labeled-pixel list (0..{n-1}).")
if (set(train_indices) & set(val_indices)) or (set(train_indices) & set(test_indices)) or (set(val_indices) & set(test_indices)):
    raise ValueError("Spatial split indices overlap.")
print("Cube:", cube.shape, "GT:", gt.shape, "labeled centers:", n)
print("Split sizes:", len(train_indices), len(val_indices), len(test_indices))


Cube: (512, 217, 204) GT: (512, 217) labeled centers: 54129
Split sizes: 32337 10952 10840


In [3]:
# 3. PCA-30 and lazy centered-patch datasets
# Fit PCA on the full scene to match the previous PCA-30 notebook setup.
H, W, C = cube.shape
pca = PCA(n_components=PCA_COMPONENTS, svd_solver="randomized", random_state=SEED)
cube_pca = pca.fit_transform(cube.reshape(-1, C)).astype(np.float32).reshape(H, W, PCA_COMPONENTS)
print("PCA cube:", cube_pca.shape, "explained variance:", float(pca.explained_variance_ratio_.sum()))

class SalinasSpatialPatchDataset(Dataset):
    def __init__(self, cube_hwc, labeled_rows, labeled_cols, labeled_labels, indices, patch_size):
        self.rows = labeled_rows[np.asarray(indices, dtype=np.int64)]
        self.cols = labeled_cols[np.asarray(indices, dtype=np.int64)]
        self.labels = labeled_labels[np.asarray(indices, dtype=np.int64)]
        self.patch_size = int(patch_size)
        self.radius = self.patch_size // 2
        self.padded = np.pad(cube_hwc, ((self.radius, self.radius), (self.radius, self.radius), (0, 0)), mode="reflect")
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        r, c = int(self.rows[index]), int(self.cols[index])
        patch = self.padded[r:r+self.patch_size, c:c+self.patch_size, :]
        patch = np.ascontiguousarray(patch.transpose(2, 0, 1), dtype=np.float32)
        return torch.from_numpy(patch), torch.tensor(int(self.labels[index]), dtype=torch.long)

train_ds = SalinasSpatialPatchDataset(cube_pca, rows, cols, labels_all, train_indices, PATCH_SIZE)
val_ds = SalinasSpatialPatchDataset(cube_pca, rows, cols, labels_all, val_indices, PATCH_SIZE)
test_ds = SalinasSpatialPatchDataset(cube_pca, rows, cols, labels_all, test_indices, PATCH_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
xb0, yb0 = next(iter(train_loader))
print("Batch:", tuple(xb0.shape), "labels:", tuple(yb0.shape))


PCA cube: (512, 217, 30) explained variance: 0.9999034404754639
Batch: (128, 30, 15, 15) labels: (128,)


In [4]:
# 4. Clone and import the official SSFCT implementation
REPO_URL = "https://github.com/SQlll-lang/NN-SSFCT-2025.git"
REPO_DIR = PROJECT_ROOT / "outputs" / "external" / "NN-SSFCT-2025"
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / "models.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
repo_str = str(REPO_DIR.resolve())
if repo_str in sys.path:
    sys.path.remove(repo_str)
sys.path.insert(0, repo_str)

try:
    import einops, timm
except ImportError as exc:
    raise ImportError("SSFCT dependencies are missing. Install the repository requirements (especially einops and timm), then restart the kernel.") from exc

from models import SSFCT
model = SSFCT(channels=PCA_COMPONENTS, n_classes=16, patch_size=PATCH_SIZE).to(DEVICE)
with torch.no_grad():
    test_logits = model(xb0.to(DEVICE).unsqueeze(1))
print("Output shape:", tuple(test_logits.shape))
assert tuple(test_logits.shape) == (len(xb0), 16), "SSFCT output shape mismatch"
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Output shape: (128, 16)
Trainable parameters: 1163399


In [5]:
# 5. Optimizer, checkpoint paths, and train/validation helpers
MODEL_TAG = "ssfct_pca30_spatial_bs128_fixed50"
LAST_CKPT = OUTPUT_DIR / f"{MODEL_TAG}_last.pth"
BEST_CKPT = OUTPUT_DIR / f"{MODEL_TAG}_best.pth"
HISTORY_CSV = OUTPUT_DIR / f"{MODEL_TAG}_history.csv"
RESULT_JSON = OUTPUT_DIR / f"{MODEL_TAG}_test_metrics.json"
REPORT_TXT = OUTPUT_DIR / f"{MODEL_TAG}_classification_report.txt"
CM_CSV = OUTPUT_DIR / f"{MODEL_TAG}_confusion_matrix.csv"

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, training, show_progress=True, progress_desc=None):
    """Run one epoch and preserve the notebook's existing metric definitions."""
    model.train(training)
    losses, ys, preds = [], [], []
    context = torch.enable_grad() if training else torch.no_grad()
    iterator = loader
    if show_progress:
        iterator = tqdm(loader, desc=progress_desc or ("Train" if training else "Validation"), leave=False)

    with context:
        for xb, yb in iterator:
            # Dataset returns [B, PCA_COMPONENTS, H, W]. SSFCT expects [B, 1, C, H, W].
            xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True).unsqueeze(1)
            yb = yb.to(DEVICE, dtype=torch.long, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                optimizer.step()
            losses.append(float(loss.detach().item()))
            ys.append(yb.detach().cpu().numpy())
            preds.append(logits.argmax(1).detach().cpu().numpy())
            if show_progress:
                iterator.set_postfix(loss=f"{loss.item():.4f}")

    if not ys:
        raise RuntimeError("The data loader produced no batches.")
    yt, yp = np.concatenate(ys), np.concatenate(preds)
    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "y_true": yt,
        "y_pred": yp,
    }

history, best_val_f1, start_epoch = [], -1.0, 1
if LAST_CKPT.exists():
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    history = ckpt.get("history", [])
    best_val_f1 = float(ckpt.get("best_val_macro_f1", -1.0))
    start_epoch = int(ckpt["epoch"]) + 1
    print(f"Resuming from epoch {start_epoch}; best validation Macro-F1={best_val_f1:.4f}")
else:
    print("Starting a fresh run.")


C:\Users\Dines\AppData\Local\Temp\ipykernel_26340\3272274342.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(LAST_CKPT, map_location=DEVICE)


Resuming from epoch 6; best validation Macro-F1=0.9479


In [ ]:
# 6. Fixed 50-epoch training (no early stopping)
# Batch-level progress is shown for train and validation. Checkpoints are saved after each completed epoch.
for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.perf_counter()
    train_m = run_epoch(train_loader, training=True, show_progress=True,
                        progress_desc=f"Epoch {epoch:02d}/{EPOCHS} train")
    val_m = run_epoch(val_loader, training=False, show_progress=True,
                      progress_desc=f"Epoch {epoch:02d}/{EPOCHS} val")
    scheduler.step()
    seconds = time.perf_counter() - t0
    row = {"epoch": epoch, "train_loss": train_m["loss"], "train_accuracy": train_m["accuracy"],
           "train_macro_f1": train_m["macro_f1"], "val_loss": val_m["loss"],
           "val_accuracy": val_m["accuracy"], "val_macro_f1": val_m["macro_f1"],
           "learning_rate": optimizer.param_groups[0]["lr"], "epoch_seconds": seconds}
    history = [h for h in history if int(h["epoch"]) != epoch] + [row]
    pd.DataFrame(history).sort_values("epoch").to_csv(HISTORY_CSV, index=False)

    if val_m["macro_f1"] > best_val_f1:
        best_val_f1 = val_m["macro_f1"]
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "best_val_macro_f1": best_val_f1,
                    "config": {"seed": SEED, "patch_size": PATCH_SIZE, "pca_components": PCA_COMPONENTS,
                               "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LEARNING_RATE,
                               "weight_decay": WEIGHT_DECAY, "split_file": str(SPLIT_PATH),
                               "repo": REPO_URL}}, BEST_CKPT)

    torch.save({"epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                "best_val_macro_f1": best_val_f1, "history": history,
                "config": {"seed": SEED, "patch_size": PATCH_SIZE, "pca_components": PCA_COMPONENTS,
                           "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LEARNING_RATE,
                           "weight_decay": WEIGHT_DECAY, "split_file": str(SPLIT_PATH),
                           "repo": REPO_URL}}, LAST_CKPT)
    tqdm.write(f"Epoch {epoch:02d}/{EPOCHS} | train loss={train_m['loss']:.4f} acc={train_m['accuracy']:.4f} F1={train_m['macro_f1']:.4f} | val loss={val_m['loss']:.4f} acc={val_m['accuracy']:.4f} F1={val_m['macro_f1']:.4f} | {seconds/60:.1f} min")
print("Training loop finished.")


Epoch 06/50 | train loss=0.0001 acc=1.0000 F1=1.0000 | val loss=0.9201 acc=0.8906 F1=0.9390 | 3.7 min


Epoch 07/50 | train loss=0.0013 acc=0.9996 F1=0.9998 | val loss=1.0065 acc=0.8945 F1=0.9387 | 4.5 min


Epoch 08/50 | train loss=0.0148 acc=0.9973 F1=0.9931 | val loss=0.6974 acc=0.8891 F1=0.9156 | 5.2 min


Epoch 09/50 | train loss=0.0013 acc=0.9996 F1=0.9998 | val loss=0.8224 acc=0.8966 F1=0.9320 | 5.7 min


Epoch 10/50 | train loss=0.0002 acc=1.0000 F1=1.0000 | val loss=1.0758 acc=0.8888 F1=0.9274 | 5.6 min


Epoch 11/50 | train loss=0.0001 acc=1.0000 F1=1.0000 | val loss=0.8312 acc=0.9030 F1=0.9349 | 5.5 min


Epoch 12/50 | train loss=0.0000 acc=1.0000 F1=1.0000 | val loss=1.0888 acc=0.8943 F1=0.9314 | 5.8 min


Epoch 13/50 | train loss=0.0000 acc=1.0000 F1=1.0000 | val loss=1.0563 acc=0.8974 F1=0.9327 | 6.0 min


Epoch 14/50 train:   6%|▋         | 16/253 [00:20<04:56,  1.25s/it, loss=0.0000]

In [ ]:
# 7. Test evaluation using the best validation Macro-F1 checkpoint
if not BEST_CKPT.exists():
    raise FileNotFoundError("No best checkpoint found. Run training first.")
best = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(best["model_state"])
test_m = run_epoch(test_loader, training=False)
y_true, y_pred = test_m["y_true"], test_m["y_pred"]

class_names = [
    "Brocoli_green_weeds_1", "Brocoli_green_weeds_2", "Fallow", "Fallow_rough_plow",
    "Fallow_smooth", "Stubble", "Celery", "Grapes_untrained", "Soil_vinyard_develop",
    "Corn_senesced_green_weeds", "Lettuce_romaine_4wk", "Lettuce_romaine_5wk",
    "Lettuce_romaine_6wk", "Lettuce_romaine_7wk", "Vinyard_untrained", "Vinyard_vertical_trellis"
]
report = classification_report(y_true, y_pred, labels=np.arange(16), target_names=class_names, digits=4, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(16))
metrics = {"model": "ssfct", "accuracy": float(accuracy_score(y_true, y_pred)),
           "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
           "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
           "kappa": float(cohen_kappa_score(y_true, y_pred)), "best_epoch": int(best["epoch"]),
           "best_val_macro_f1": float(best["best_val_macro_f1"]), "train_samples": len(train_ds),
           "val_samples": len(val_ds), "test_samples": len(test_ds), "seed": SEED,
           "patch_size": PATCH_SIZE, "pca_components": PCA_COMPONENTS, "batch_size": BATCH_SIZE,
           "epochs_configured": EPOCHS, "split_file": str(SPLIT_PATH)}
RESULT_JSON.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
REPORT_TXT.write_text(report, encoding="utf-8")
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(CM_CSV)
print(json.dumps(metrics, indent=2))
print(report)
print("Saved test metrics, class report, and confusion matrix.")


In [ ]:
# 8. Efficiency benchmark in the shared GAST-style schema
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

model.eval()
example_x, _ = next(iter(test_loader))
example_x = example_x.to(DEVICE).unsqueeze(1)
params = sum(p.numel() for p in model.parameters())
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
model_size_mb = (param_bytes + buffer_bytes) / (1024**2)

flops = np.nan
try:
    from thop import profile
    flops, _ = profile(model, inputs=(example_x[:1],), verbose=False)
    flops = float(flops)
except Exception as exc:
    print("FLOPs unavailable:", repr(exc))

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
latencies = []
with torch.inference_mode():
    for _ in range(WARMUP_RUNS):
        _ = model(example_x)
    sync_device()
    for _ in range(TIMED_RUNS):
        sync_device()
        t0 = time.perf_counter()
        _ = model(example_x)
        sync_device()
        latencies.append((time.perf_counter() - t0) * 1000)
latencies = np.asarray(latencies, dtype=np.float64)
lat_mean = float(latencies.mean())
peak_mem = float(torch.cuda.max_memory_allocated() / (1024**2)) if DEVICE.type == "cuda" else np.nan

efficiency_row = {
    "model": "ssfct", "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
    "parameters": int(params), "parameters_m": float(params / 1e6),
    "model_size_mb": float(model_size_mb), "flops": flops,
    "gflops": float(flops / 1e9) if np.isfinite(flops) else np.nan,
    "peak_gpu_memory_mb": peak_mem, "batch_size": int(BATCH_SIZE),
    "latency_batch_ms": lat_mean, "latency_batch_median_ms": float(np.median(latencies)),
    "latency_batch_p95_ms": float(np.percentile(latencies, 95)),
    "latency_batch_std_ms": float(latencies.std(ddof=0)),
    "latency_per_sample_ms": float(lat_mean / BATCH_SIZE),
    "throughput_samples_sec": float(BATCH_SIZE / (lat_mean / 1000)),
    "warmup_runs": int(WARMUP_RUNS), "timed_runs": int(TIMED_RUNS)
}
columns = ["model","accuracy","macro_f1","parameters","parameters_m","model_size_mb","flops","gflops",
           "peak_gpu_memory_mb","batch_size","latency_batch_ms","latency_batch_median_ms",
           "latency_batch_p95_ms","latency_batch_std_ms","latency_per_sample_ms",
           "throughput_samples_sec","warmup_runs","timed_runs"]
single_csv = OUTPUT_DIR / f"{MODEL_TAG}_efficiency.csv"
pd.DataFrame([efficiency_row], columns=columns).to_csv(single_csv, index=False)

shared_csv = OUTPUT_DIR / "efficiency_results.csv"
new_df = pd.DataFrame([efficiency_row], columns=columns)
if shared_csv.exists():
    try:
        old_df = pd.read_csv(shared_csv)
        if not set(columns).issubset(old_df.columns):
            old_df = pd.DataFrame(columns=columns)
        else:
            old_df = old_df[columns]
        old_df = old_df[old_df["model"].astype(str).str.lower() != "ssfct"]
        new_df = pd.concat([old_df, new_df], ignore_index=True)
    except Exception:
        pass
new_df.to_csv(shared_csv, index=False)
print(new_df.tail(1).to_string(index=False))
print("Saved:", single_csv)
print("Updated:", shared_csv)


## Files saved under `outputs/salinas/`

- `ssfct_pca30_spatial_bs128_fixed50_last.pth` — last completed epoch and optimizer/scheduler state for resuming
- `ssfct_pca30_spatial_bs128_fixed50_best.pth` — best validation Macro-F1 checkpoint
- `ssfct_pca30_spatial_bs128_fixed50_history.csv` — per-epoch training/validation history
- `ssfct_pca30_spatial_bs128_fixed50_test_metrics.json` — held-out test metrics
- `ssfct_pca30_spatial_bs128_fixed50_classification_report.txt` — per-class precision/recall/F1
- `ssfct_pca30_spatial_bs128_fixed50_confusion_matrix.csv` — confusion matrix
- `ssfct_pca30_spatial_bs128_fixed50_efficiency.csv` — efficiency row
- `efficiency_results.csv` — shared efficiency table

**Reproducibility notes:** The split file is checked for valid index ranges and overlap. The notebook fits PCA on the full scene, matching the previous PCA-30 setup (unsupervised full-scene preprocessing). The SSFCT network comes from the official repository; data loading, spatial split, training configuration, and benchmarking are adapted for this experiment. Latency is inference-only at the configured batch size. FLOPs may remain unavailable if `thop` cannot trace the model.